Question 1 — Data Modeling
Entities & Relationships
Student: Represents a person enrolled at the school.

Course: Represents a subject taught at the school.

Enrollment: A bridge entity mapping a student to a specific course (Handles the Many-to-Many relationship).

Models & Relationships
Entity: Student

Attributes: student_id, name, email, level

Entity: Course

Attributes: course_id, title, credits, department

Entity: Enrollment

Attributes: enrollment_id, student_id, course_id, enrollment_date

Relationships:

One Student can have many Enrollments (1:N).

One Course can have many Enrollments (1:N).

TASK 2 LOGICAL MODEL
Student              Course               Enrollment
----------           ----------           ---------------
student_id (PK)      course_id (PK)       enrollment_id (PK)
name                 title                student_id (FK)
email                credits              course_id (FK)
level                department           enrollment_date

In [1]:
#PYDANTIC MODEL
from pydantic import BaseModel
from datetime import date

class Student(BaseModel):
    student_id: int
    name: str
    email: str
    level: int

class Course(BaseModel):
    course_id: int
    title: str
    credits: int
    department: str

class Enrollment(BaseModel):
    enrollment_id: int
    student_id: int
    course_id: int
    enrollment_date: date

In [2]:
#QUESTION 3
from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float
    quantity: int
    in_stock: bool

# Instantiate with sample data
product_instance = Product(
    name="Wireless Mouse",
    price=29.99,
    quantity=150,
    in_stock=True
)

print(product_instance)

name='Wireless Mouse' price=29.99 quantity=150 in_stock=True


In [3]:
#QUESTION 4
product = Product(
    name="Laptop",
    price="1200.50",
    quantity="5",
    in_stock="True"
)
print(product)

name='Laptop' price=1200.5 quantity=5 in_stock=True


Prediction & ExplanationPrediction: The code runs successfully without raising errors.Why it happens: Pydantic performs Type Coercion. If data is passed as a string but can be parsed cleanly into the designated target type , Pydantic auto-converts it behind the scenes.

In [ ]:
#QUESTION 5
from pydantic import BaseModel
from typing import Optional

class User(BaseModel):
    name: str                  # Required
    email: str                 # Required
    age: Optional[int] = None  # Optional
    active: bool = True        # Default Value

# 1. Valid instance with all fields
user_full = User(name="Alice", email="alice@example.com", age=25, active=False)

# 2. Instance omitting optional fields
user_omitted = User(name="Bob", email="bob@example.com")

print("Full User:", user_full)
print("Omitted User:", user_omitted)

Full User: name='Alice' email='alice@example.com' age=25 active=False
Omitted User: name='Bob' email='bob@example.com' age=None active=True


Explanation: When age is omitted, it falls back to its default value of None. When active is omitted, it safely defaults to True. name and email must be provided, or Pydantic will throw a ValidationError

In [ ]:
#QUESTION 6
from pydantic import BaseModel, Field, field_validator, ValidationError

class StudentModel(BaseModel):
    name: str = Field(..., min_length=3)
    age: int = Field(..., ge=16, le=60)
    email: str

    @field_validator('email')
    @classmethod
    def check_at_sign(cls, v: str) -> str:
        if '@' not in v:
            raise ValueError('Email must contain an @ symbol')
        return v

# --- Test Cases ---

# 1. Valid Input
try:
    s1 = StudentModel(name="John", age=20, email="john@school.com")
    print("Valid Student:", s1)
except ValidationError as e:
    print(e)

# 2. Invalid Age
try:
    s2 = StudentModel(name="Alex", age=12, email="alex@school.com")
except ValidationError as e:
    print("\nInvalid Age Error:\n", e)

# 3. Invalid Email
try:
    s3 = StudentModel(name="Sam", age=22, email="samschool.com")
except ValidationError as e:
    print("\nInvalid Email Error:\n", e)

Valid Student: name='John' age=20 email='john@school.com'

Invalid Age Error:
 1 validation error for StudentModel
age
  Input should be greater than or equal to 16 [type=greater_than_equal, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal

Invalid Email Error:
 1 validation error for StudentModel
email
  Value error, Email must contain an @ symbol [type=value_error, input_value='samschool.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


Explanation of Results
Valid Input: Successfully creates the object because all criteria are matched.

Invalid Age: Throws a ValidationError because 12 falls outside the explicit bounds configuration of ge=16 (greater than or equal to 16).

Invalid Email: Throws a custom ValidationError powered by our @field_validator, blocking any input lacking an @ symbol.

In [6]:
#QUESTION 7
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    state: str
    country: str

class UserWithAddress(BaseModel):
    name: str
    email: str
    address: Address

# Nested JSON-like data
user_data = {
    "name": "Jane Doe",
    "email": "jane@example.com",
    "address": {
        "city": "Austin",
        "state": "Texas",
        "country": "USA"
    }
}

# Instantiate
user_obj = UserWithAddress(**user_data)

# Serialize using model_dump()
serialized_dict = user_obj.model_dump()
print(serialized_dict)

{'name': 'Jane Doe', 'email': 'jane@example.com', 'address': {'city': 'Austin', 'state': 'Texas', 'country': 'USA'}}


In [7]:
#QUESTION 8
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List, Optional
from datetime import date

# 1. Course Model
class Course(BaseModel):
    course_id: str = Field(..., min_length=3, max_length=10)
    title: str = Field(..., min_length=2)
    credits: int = Field(..., ge=1, le=5)

# 2. Student Model
class Student(BaseModel):
    student_id: int
    name: str = Field(..., min_length=2)
    email: str
    major: Optional[str] = "Undeclared"

    @field_validator('email')
    @classmethod
    def validate_edu_email(cls, v: str) -> str:
        if not v.endswith('.edu'):
            raise ValueError("Student enrollment requires a valid institutional '.edu' email address.")
        return v

# 3. Enrollment Model (Nested Integration)
class Enrollment(BaseModel):
    enrollment_id: str
    student: Student
    course: Course
    enrollment_date: date = Field(default_factory=date.today)


# --- Execution and Verification Check ---
if __name__ == "__main__":
    try:
        # Build Valid Components
        math_course = Course(course_id="MATH101", title="Calculus I", credits=4)
        charlie_student = Student(student_id=4002, name="Charlie", email="charlie@university.edu")
        
        # Build Integrated Model
        active_enrollment = Enrollment(enrollment_id="ENR-2026-001", student=charlie_student, course=math_course)
        
        print("--- Integrated Enrollment Successfully Constructed ---")
        print(active_enrollment.model_dump_json(indent=2))

    except ValidationError as e:
        print("Validation Failed:", e)

--- Integrated Enrollment Successfully Constructed ---
{
  "enrollment_id": "ENR-2026-001",
  "student": {
    "student_id": 4002,
    "name": "Charlie",
    "email": "charlie@university.edu",
    "major": "Undeclared"
  },
  "course": {
    "course_id": "MATH101",
    "title": "Calculus I",
    "credits": 4
  },
  "enrollment_date": "2026-06-29"
}
